# HAM Synthetic Experiments Runner

Run this notebook on a GPU to massively accelerate JAX JIT compilation and the multi-iteration inverse problem tests.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure the project src/ is in the python path
project_root = Path(os.getcwd()).parent.parent.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import jax

print("Available JAX devices:", jax.devices())

In [ ]:
from experiments.wildfire.synthetic.run_experiments import get_all_experiments

# ==================================================================
# Configuration
# ==================================================================
categories_to_run = ["A", "B", "C", "D", "E"]  # Select categories, e.g. ['C', 'E']
experiments_to_run = []  # Specific experiment names, e.g. ['C1'] (overrides categories if not empty)

quick_mode = False  # True = smaller grids for rapid prototyping
save_results = True  # Save metrics to summary.json
visualize_results = True  # Save PDFs of metric recoveries
verbose = True  # Print solver logs
# ==================================================================

In [ ]:
all_experiments = get_all_experiments()

for category, exp_list in all_experiments.items():
    for exp_id, exp_class, kwargs, quick_kwargs in exp_list:
        should_run = False
        if len(experiments_to_run) > 0:
            should_run = any(e.upper() == exp_id.upper() for e in experiments_to_run)
        else:
            should_run = category.upper() in categories_to_run

        if should_run:
            print(
                f"\n{'=' * 60}\nExperiment: {exp_id} - {exp_class.__name__}\nCategory: {category}\n{'=' * 60}"
            )

            # Initialize with appropriate kwargs
            exp_kwargs = quick_kwargs if quick_mode else kwargs
            exp = exp_class(**exp_kwargs)

            # Execute
            try:
                result = exp.execute(
                    save=save_results, visualize=visualize_results, verbose=verbose
                )
                print(f"\nSuccess: {result.success}")
                print("Metrics:")
                for k, v in result.metrics.items():
                    print(f"  {k}: {v:.6f}")
            except Exception as e:
                print(f"\nERROR running {exp_id}: {e}")
                import traceback

                traceback.print_exc()